# Cross-Validation with Missing Events: Visual Demonstration

This notebook demonstrates how cross-validation handles missing stimulus events across different scenarios.

**Setup:**
- 1 voxel (for clarity)
- 2 runs, 100 TRs each, TR=1s
- 4 events, each occurs 2x per run (8 onsets total if present)
- Events are 1s duration, convolved with canonical HRF
- Simple LORO CV: train on run 1, test on run 2

**Scenarios:**
1. Baseline: All events present in both runs
2. Train missing: Event 2 missing from run 1 (train)
3. Test missing: Event 2 missing from run 2 (test)
4. Both missing different: Event 1 missing from run 2, Event 2 missing from run 1

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
from scipy.stats import gamma
from typing import Tuple, List

# Set style
plt.style.use('seaborn-v0_8-darkgrid')
%matplotlib inline

## Helper Functions

In [ ]:
def canonical_hrf(tr: float = 1.0, duration: float = 32.0) -> np.ndarray:
    """
    Create canonical HRF (SPM-style double gamma).
    
    Parameters
    ----------
    tr : float
        Repetition time in seconds
    duration : float
        Total duration in seconds
    
    Returns
    -------
    hrf : np.ndarray
        HRF sampled at TR
    """
    t = np.arange(0, duration, tr)
    
    # Parameters for double gamma
    # Positive gamma
    peak1 = 6.0
    scale1 = 1.0
    pos = gamma.pdf(t, peak1, scale=scale1)
    
    # Negative gamma (undershoot)
    peak2 = 16.0
    scale2 = 1.0
    neg = gamma.pdf(t, peak2, scale=scale2)
    
    # Combine with 6:1 ratio
    hrf = pos - neg / 6.0
    
    # Normalize to peak at 1
    hrf = hrf / hrf.max()
    
    return hrf


def create_event_design(
    n_timepoints: int,
    event_onsets: List[int],
    tr: float = 1.0,
    duration: float = 1.0,
) -> np.ndarray:
    """
    Create HRF-convolved design matrix for a single event type.
    
    Parameters
    ----------
    n_timepoints : int
        Number of timepoints
    event_onsets : List[int]
        Onset times in TRs
    tr : float
        Repetition time
    duration : float
        Event duration in seconds
    
    Returns
    -------
    design : np.ndarray
        Convolved design (n_timepoints,)
    """
    # Create stick function
    stick = np.zeros(n_timepoints)
    for onset in event_onsets:
        if 0 <= onset < n_timepoints:
            # Set amplitude = duration (in TRs)
            stick[onset] = duration / tr
    
    # Convolve with HRF
    hrf = canonical_hrf(tr=tr, duration=32.0)
    convolved = np.convolve(stick, hrf, mode='full')[:n_timepoints]
    
    return convolved


def create_full_design(
    n_timepoints: int,
    n_events: int,
    events_per_run: int = 2,
    missing_events: List[int] = None,
    seed: int = 42,
) -> Tuple[np.ndarray, List[List[int]]]:
    """
    Create design matrix with multiple events.
    
    Parameters
    ----------
    n_timepoints : int
        Number of timepoints
    n_events : int
        Number of event types
    events_per_run : int
        How many times each event occurs
    missing_events : List[int]
        Which events to zero out (make missing)
    seed : int
        Random seed for onset timing
    
    Returns
    -------
    design : np.ndarray
        Design matrix (n_timepoints, n_events)
    onset_times : List[List[int]]
        Onset times for each event
    """
    if missing_events is None:
        missing_events = []
    
    rng = np.random.RandomState(seed)
    design = np.zeros((n_timepoints, n_events))
    onset_times = []
    
    for event_idx in range(n_events):
        if event_idx in missing_events:
            # Event is missing - leave as zeros
            onset_times.append([])
        else:
            # Generate evenly spaced onsets with jitter
            spacing = n_timepoints // (events_per_run + 1)
            onsets = []
            for i in range(events_per_run):
                # Base position
                base = (i + 1) * spacing
                # Add jitter (±20% of spacing)
                jitter = rng.randint(-spacing//5, spacing//5 + 1)
                onset = max(5, min(n_timepoints - 10, base + jitter))
                onsets.append(onset)
            
            onset_times.append(sorted(onsets))
            design[:, event_idx] = create_event_design(n_timepoints, onsets)
    
    return design, onset_times


def generate_fmri_data(
    design: np.ndarray,
    true_betas: np.ndarray,
    noise_std: float = 0.5,
    seed: int = 42,
) -> np.ndarray:
    """
    Generate synthetic fMRI data.
    
    Parameters
    ----------
    design : np.ndarray
        Design matrix (n_timepoints, n_events)
    true_betas : np.ndarray
        True beta coefficients (n_events,)
    noise_std : float
        Noise standard deviation
    seed : int
        Random seed
    
    Returns
    -------
    data : np.ndarray
        Synthetic timeseries (n_timepoints,)
    """
    rng = np.random.RandomState(seed)
    
    # Signal = design @ betas
    signal = design @ true_betas
    
    # Add noise
    noise = rng.randn(len(signal)) * noise_std
    
    data = signal + noise
    
    return data


def fit_ols(data: np.ndarray, design: np.ndarray) -> np.ndarray:
    """
    Fit OLS model: beta = (X'X)^-1 X'y
    
    Parameters
    ----------
    data : np.ndarray
        Timeseries (n_timepoints,)
    design : np.ndarray
        Design matrix (n_timepoints, n_events)
    
    Returns
    -------
    betas : np.ndarray
        Estimated betas (n_events,)
    """
    # Remove zero columns
    non_zero_cols = ~np.all(design == 0, axis=0)
    design_fit = design[:, non_zero_cols]
    
    if design_fit.shape[1] == 0:
        return np.zeros(design.shape[1])
    
    # Fit
    betas_fit = np.linalg.lstsq(design_fit, data, rcond=None)[0]
    
    # Reconstruct full beta vector
    betas_full = np.zeros(design.shape[1])
    betas_full[non_zero_cols] = betas_fit
    
    return betas_full


def predict(
    betas: np.ndarray,
    design: np.ndarray,
    strategy: str = 'zero',
) -> np.ndarray:
    """
    Predict timeseries using fitted betas.
    
    Parameters
    ----------
    betas : np.ndarray
        Fitted betas (n_events,)
    design : np.ndarray
        Test design matrix (n_timepoints, n_events)
    strategy : str
        'zero' or 'nuisance' (for nuisance, need to implement projection)
    
    Returns
    -------
    prediction : np.ndarray
        Predicted timeseries (n_timepoints,)
    """
    # For this simple case, just use betas as-is
    # (zero strategy is implicit - zero betas for missing events)
    prediction = design @ betas
    
    return prediction


def compute_r2(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    """
    Compute coefficient of determination (R²).
    
    R² = 1 - SS_res / SS_tot
    """
    ss_res = np.sum((y_true - y_pred) ** 2)
    ss_tot = np.sum((y_true - y_true.mean()) ** 2)
    
    r2 = 1 - (ss_res / ss_tot)
    
    return r2

## Setup: Common Parameters

In [ ]:
# Parameters
n_timepoints = 100  # TRs per run
n_events = 4  # Event types
events_per_run = 2  # Each event occurs 2x per run
tr = 1.0  # seconds
noise_std = 0.5  # Noise level

# True betas (same for both runs!)
true_betas = np.array([2.0, 1.5, 1.0, 0.5])  # Event 0 strongest, Event 3 weakest

print(f"True betas: {true_betas}")
print(f"Events per run: {events_per_run}")
print(f"Total onsets per event (if present): {events_per_run}")

## Scenario 1: Baseline - All Events Present

All 4 events occur in both runs. This is the ideal case - we should get good prediction.

In [ ]:
# Create design matrices
design_train, onsets_train = create_full_design(
    n_timepoints, n_events, events_per_run, missing_events=[], seed=42
)
design_test, onsets_test = create_full_design(
    n_timepoints, n_events, events_per_run, missing_events=[], seed=43
)

# Generate data
data_train = generate_fmri_data(design_train, true_betas, noise_std, seed=100)
data_test = generate_fmri_data(design_test, true_betas, noise_std, seed=101)

# Fit on train
betas_fit = fit_ols(data_train, design_train)

# Predict on test
prediction = predict(betas_fit, design_test)

# Compute R²
r2 = compute_r2(data_test, prediction)

print(f"True betas:   {true_betas}")
print(f"Fitted betas: {betas_fit}")
print(f"R² = {r2:.4f}")

In [ ]:
# Plot
fig, axes = plt.subplots(3, 1, figsize=(14, 10))
time = np.arange(n_timepoints) * tr
event_colors = ['#e74c3c', '#3498db', '#2ecc71', '#f39c12']
event_names = ['Event 0', 'Event 1', 'Event 2', 'Event 3']

# Panel 1: Train data + design
ax = axes[0]
ax.plot(time, data_train, 'k-', linewidth=2, label='Train Data', alpha=0.7)
ax.plot(time, design_train @ true_betas, 'k--', linewidth=1, label='True Signal', alpha=0.5)

# Add onset markers
y_min, y_max = ax.get_ylim()
for event_idx in range(n_events):
    for onset in onsets_train[event_idx]:
        ax.axvline(onset * tr, color=event_colors[event_idx], alpha=0.3, linestyle='-', linewidth=2)

ax.set_xlabel('Time (s)', fontsize=12)
ax.set_ylabel('Signal', fontsize=12)
ax.set_title('Train Run: Data and Event Onsets', fontsize=14, fontweight='bold')
ax.legend(loc='upper right')
ax.grid(True, alpha=0.3)

# Panel 2: Test data
ax = axes[1]
ax.plot(time, data_test, 'k-', linewidth=2, label='Test Data', alpha=0.7)
ax.plot(time, design_test @ true_betas, 'k--', linewidth=1, label='True Signal', alpha=0.5)

# Add onset markers
for event_idx in range(n_events):
    for onset in onsets_test[event_idx]:
        ax.axvline(onset * tr, color=event_colors[event_idx], alpha=0.3, linestyle='-', linewidth=2)

ax.set_xlabel('Time (s)', fontsize=12)
ax.set_ylabel('Signal', fontsize=12)
ax.set_title('Test Run: Data and Event Onsets', fontsize=14, fontweight='bold')
ax.legend(loc='upper right')
ax.grid(True, alpha=0.3)

# Panel 3: Prediction
ax = axes[2]
ax.plot(time, data_test, 'k-', linewidth=2, label='Test Data', alpha=0.5)
ax.plot(time, prediction, 'r-', linewidth=2, label='Prediction', alpha=0.8)
ax.plot(time, data_test - prediction, 'gray', linewidth=1, label='Residual', alpha=0.5)

ax.set_xlabel('Time (s)', fontsize=12)
ax.set_ylabel('Signal', fontsize=12)
ax.set_title(f'Prediction vs. Test Data (R² = {r2:.4f})', fontsize=14, fontweight='bold')
ax.legend(loc='upper right')
ax.grid(True, alpha=0.3)

# Add legend for event colors
from matplotlib.patches import Patch
legend_elements = [Patch(facecolor=event_colors[i], alpha=0.5, label=f'{event_names[i]} (β={true_betas[i]:.1f})') 
                   for i in range(n_events)]
fig.legend(handles=legend_elements, loc='upper center', ncol=4, fontsize=10, 
           bbox_to_anchor=(0.5, 0.98))

plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.show()

## Scenario 2: Train Missing Event 2

Event 2 is present in the test run but **missing** from the train run.
- We cannot learn a beta for Event 2 (no data to fit)
- Fitted beta for Event 2 will be 0
- Test data contains Event 2 signal, but we can't predict it
- R² will be lower

In [ ]:
# Create design matrices - Event 2 missing from train
design_train, onsets_train = create_full_design(
    n_timepoints, n_events, events_per_run, missing_events=[2], seed=42
)
design_test, onsets_test = create_full_design(
    n_timepoints, n_events, events_per_run, missing_events=[], seed=43
)

# Generate data
data_train = generate_fmri_data(design_train, true_betas, noise_std, seed=100)
data_test = generate_fmri_data(design_test, true_betas, noise_std, seed=101)

# Fit on train
betas_fit = fit_ols(data_train, design_train)

# Predict on test
prediction = predict(betas_fit, design_test)

# Compute R²
r2 = compute_r2(data_test, prediction)

print(f"True betas:   {true_betas}")
print(f"Fitted betas: {betas_fit}")
print(f"Missing:      Event 2 (train)")
print(f"R² = {r2:.4f}")

In [ ]:
# Plot
fig, axes = plt.subplots(3, 1, figsize=(14, 10))
time = np.arange(n_timepoints) * tr

# Panel 1: Train data + design
ax = axes[0]
ax.plot(time, data_train, 'k-', linewidth=2, label='Train Data', alpha=0.7)
ax.plot(time, design_train @ true_betas, 'k--', linewidth=1, label='True Signal', alpha=0.5)

# Add onset markers (Event 2 missing!)
for event_idx in range(n_events):
    for onset in onsets_train[event_idx]:
        ax.axvline(onset * tr, color=event_colors[event_idx], alpha=0.3, linestyle='-', linewidth=2)

ax.text(0.02, 0.95, '⚠️ Event 2 MISSING', transform=ax.transAxes, 
        fontsize=12, fontweight='bold', color=event_colors[2],
        verticalalignment='top', bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

ax.set_xlabel('Time (s)', fontsize=12)
ax.set_ylabel('Signal', fontsize=12)
ax.set_title('Train Run: Event 2 Missing (No onsets shown)', fontsize=14, fontweight='bold')
ax.legend(loc='upper right')
ax.grid(True, alpha=0.3)

# Panel 2: Test data
ax = axes[1]
ax.plot(time, data_test, 'k-', linewidth=2, label='Test Data', alpha=0.7)
ax.plot(time, design_test @ true_betas, 'k--', linewidth=1, label='True Signal', alpha=0.5)

# Add onset markers (Event 2 present!)
for event_idx in range(n_events):
    style = '-' if event_idx == 2 else '-'
    alpha = 0.6 if event_idx == 2 else 0.3
    for onset in onsets_test[event_idx]:
        ax.axvline(onset * tr, color=event_colors[event_idx], alpha=alpha, linestyle=style, linewidth=3 if event_idx==2 else 2)

ax.text(0.02, 0.95, '✓ Event 2 PRESENT', transform=ax.transAxes, 
        fontsize=12, fontweight='bold', color=event_colors[2],
        verticalalignment='top', bbox=dict(boxstyle='round', facecolor='lightgreen', alpha=0.5))

ax.set_xlabel('Time (s)', fontsize=12)
ax.set_ylabel('Signal', fontsize=12)
ax.set_title('Test Run: Event 2 Present (highlighted)', fontsize=14, fontweight='bold')
ax.legend(loc='upper right')
ax.grid(True, alpha=0.3)

# Panel 3: Prediction
ax = axes[2]
ax.plot(time, data_test, 'k-', linewidth=2, label='Test Data', alpha=0.5)
ax.plot(time, prediction, 'r-', linewidth=2, label='Prediction (β₂=0!)', alpha=0.8)
ax.plot(time, data_test - prediction, 'gray', linewidth=1, label='Residual', alpha=0.5)

# Highlight where Event 2 occurs in test
for onset in onsets_test[2]:
    ax.axvspan(onset*tr, (onset+10)*tr, color=event_colors[2], alpha=0.1)

ax.set_xlabel('Time (s)', fontsize=12)
ax.set_ylabel('Signal', fontsize=12)
ax.set_title(f'Prediction: Cannot predict Event 2 (R² = {r2:.4f})', fontsize=14, fontweight='bold')
ax.legend(loc='upper right')
ax.grid(True, alpha=0.3)

# Add legend
legend_elements = [Patch(facecolor=event_colors[i], alpha=0.5, 
                         label=f'{event_names[i]} (β_true={true_betas[i]:.1f}, β_fit={betas_fit[i]:.2f})') 
                   for i in range(n_events)]
fig.legend(handles=legend_elements, loc='upper center', ncol=4, fontsize=10, 
           bbox_to_anchor=(0.5, 0.98))

plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.show()

## Scenario 3: Test Missing Event 2

Event 2 is present in the train run but **missing** from the test run.
- We CAN learn a beta for Event 2 (from train data)
- BUT test data doesn't have Event 2 signal
- Fitted beta will contribute to prediction where it shouldn't
- This is where "nuisance" strategy could help (project out Event 2 from test)

In [ ]:
# Create design matrices - Event 2 missing from test
design_train, onsets_train = create_full_design(
    n_timepoints, n_events, events_per_run, missing_events=[], seed=42
)
design_test, onsets_test = create_full_design(
    n_timepoints, n_events, events_per_run, missing_events=[2], seed=43
)

# Generate data (test shouldn't have Event 2 signal)
data_train = generate_fmri_data(design_train, true_betas, noise_std, seed=100)
data_test = generate_fmri_data(design_test, true_betas, noise_std, seed=101)

# Fit on train
betas_fit = fit_ols(data_train, design_train)

# Predict on test
prediction = predict(betas_fit, design_test)

# Compute R²
r2 = compute_r2(data_test, prediction)

print(f"True betas:   {true_betas}")
print(f"Fitted betas: {betas_fit}")
print(f"Missing:      Event 2 (test)")
print(f"R² = {r2:.4f}")

In [ ]:
# Plot
fig, axes = plt.subplots(3, 1, figsize=(14, 10))
time = np.arange(n_timepoints) * tr

# Panel 1: Train data + design
ax = axes[0]
ax.plot(time, data_train, 'k-', linewidth=2, label='Train Data', alpha=0.7)
ax.plot(time, design_train @ true_betas, 'k--', linewidth=1, label='True Signal', alpha=0.5)

# Add onset markers (Event 2 present!)
for event_idx in range(n_events):
    style = '-' if event_idx == 2 else '-'
    alpha = 0.6 if event_idx == 2 else 0.3
    for onset in onsets_train[event_idx]:
        ax.axvline(onset * tr, color=event_colors[event_idx], alpha=alpha, linestyle=style, linewidth=3 if event_idx==2 else 2)

ax.text(0.02, 0.95, '✓ Event 2 PRESENT', transform=ax.transAxes, 
        fontsize=12, fontweight='bold', color=event_colors[2],
        verticalalignment='top', bbox=dict(boxstyle='round', facecolor='lightgreen', alpha=0.5))

ax.set_xlabel('Time (s)', fontsize=12)
ax.set_ylabel('Signal', fontsize=12)
ax.set_title('Train Run: Event 2 Present (can estimate β₂)', fontsize=14, fontweight='bold')
ax.legend(loc='upper right')
ax.grid(True, alpha=0.3)

# Panel 2: Test data
ax = axes[1]
ax.plot(time, data_test, 'k-', linewidth=2, label='Test Data', alpha=0.7)
ax.plot(time, design_test @ true_betas, 'k--', linewidth=1, label='True Signal', alpha=0.5)

# Add onset markers (Event 2 missing!)
for event_idx in range(n_events):
    for onset in onsets_test[event_idx]:
        ax.axvline(onset * tr, color=event_colors[event_idx], alpha=0.3, linestyle='-', linewidth=2)

ax.text(0.02, 0.95, '⚠️ Event 2 MISSING', transform=ax.transAxes, 
        fontsize=12, fontweight='bold', color=event_colors[2],
        verticalalignment='top', bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

ax.set_xlabel('Time (s)', fontsize=12)
ax.set_ylabel('Signal', fontsize=12)
ax.set_title('Test Run: Event 2 Missing (no signal, no onsets)', fontsize=14, fontweight='bold')
ax.legend(loc='upper right')
ax.grid(True, alpha=0.3)

# Panel 3: Prediction
ax = axes[2]
ax.plot(time, data_test, 'k-', linewidth=2, label='Test Data', alpha=0.5)
ax.plot(time, prediction, 'r-', linewidth=2, label='Prediction (using β₂!)', alpha=0.8)
ax.plot(time, data_test - prediction, 'gray', linewidth=1, label='Residual', alpha=0.5)

ax.set_xlabel('Time (s)', fontsize=12)
ax.set_ylabel('Signal', fontsize=12)
ax.set_title(f'Prediction: β₂ applied but Event 2 not in test (R² = {r2:.4f})', fontsize=14, fontweight='bold')
ax.legend(loc='upper right')
ax.grid(True, alpha=0.3)

# Add legend
legend_elements = [Patch(facecolor=event_colors[i], alpha=0.5, 
                         label=f'{event_names[i]} (β_true={true_betas[i]:.1f}, β_fit={betas_fit[i]:.2f})') 
                   for i in range(n_events)]
fig.legend(handles=legend_elements, loc='upper center', ncol=4, fontsize=10, 
           bbox_to_anchor=(0.5, 0.98))

plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.show()

## Scenario 4: Both Missing Different Events

Most complex case:
- Event 1 is missing from test run (present in train)
- Event 2 is missing from train run (present in test)

Combined effects:
- Can't learn β₂ (Event 2 not in train)
- Learn β₁ but shouldn't use it (Event 1 not in test)

In [ ]:
# Create design matrices
# Train: Event 2 missing
# Test: Event 1 missing
design_train, onsets_train = create_full_design(
    n_timepoints, n_events, events_per_run, missing_events=[2], seed=42
)
design_test, onsets_test = create_full_design(
    n_timepoints, n_events, events_per_run, missing_events=[1], seed=43
)

# Generate data
data_train = generate_fmri_data(design_train, true_betas, noise_std, seed=100)
data_test = generate_fmri_data(design_test, true_betas, noise_std, seed=101)

# Fit on train
betas_fit = fit_ols(data_train, design_train)

# Predict on test
prediction = predict(betas_fit, design_test)

# Compute R²
r2 = compute_r2(data_test, prediction)

print(f"True betas:   {true_betas}")
print(f"Fitted betas: {betas_fit}")
print(f"Missing:      Event 2 (train), Event 1 (test)")
print(f"R² = {r2:.4f}")

In [ ]:
# Plot
fig, axes = plt.subplots(3, 1, figsize=(14, 10))
time = np.arange(n_timepoints) * tr

# Panel 1: Train data + design
ax = axes[0]
ax.plot(time, data_train, 'k-', linewidth=2, label='Train Data', alpha=0.7)
ax.plot(time, design_train @ true_betas, 'k--', linewidth=1, label='True Signal', alpha=0.5)

# Add onset markers (Event 2 missing!)
for event_idx in range(n_events):
    for onset in onsets_train[event_idx]:
        ax.axvline(onset * tr, color=event_colors[event_idx], alpha=0.3, linestyle='-', linewidth=2)

ax.text(0.02, 0.95, '⚠️ Event 2 MISSING', transform=ax.transAxes, 
        fontsize=12, fontweight='bold', color=event_colors[2],
        verticalalignment='top', bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

ax.set_xlabel('Time (s)', fontsize=12)
ax.set_ylabel('Signal', fontsize=12)
ax.set_title('Train Run: Event 2 Missing (cannot estimate β₂)', fontsize=14, fontweight='bold')
ax.legend(loc='upper right')
ax.grid(True, alpha=0.3)

# Panel 2: Test data
ax = axes[1]
ax.plot(time, data_test, 'k-', linewidth=2, label='Test Data', alpha=0.7)
ax.plot(time, design_test @ true_betas, 'k--', linewidth=1, label='True Signal', alpha=0.5)

# Add onset markers (Event 1 missing!)
for event_idx in range(n_events):
    for onset in onsets_test[event_idx]:
        ax.axvline(onset * tr, color=event_colors[event_idx], alpha=0.3, linestyle='-', linewidth=2)

ax.text(0.02, 0.95, '⚠️ Event 1 MISSING', transform=ax.transAxes, 
        fontsize=12, fontweight='bold', color=event_colors[1],
        verticalalignment='top', bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

ax.set_xlabel('Time (s)', fontsize=12)
ax.set_ylabel('Signal', fontsize=12)
ax.set_title('Test Run: Event 1 Missing (shouldn\'t use β₁)', fontsize=14, fontweight='bold')
ax.legend(loc='upper right')
ax.grid(True, alpha=0.3)

# Panel 3: Prediction
ax = axes[2]
ax.plot(time, data_test, 'k-', linewidth=2, label='Test Data', alpha=0.5)
ax.plot(time, prediction, 'r-', linewidth=2, label='Prediction', alpha=0.8)
ax.plot(time, data_test - prediction, 'gray', linewidth=1, label='Residual', alpha=0.5)

ax.set_xlabel('Time (s)', fontsize=12)
ax.set_ylabel('Signal', fontsize=12)
ax.set_title(f'Prediction: Missing β₂ AND wrongly using β₁ (R² = {r2:.4f})', fontsize=14, fontweight='bold')
ax.legend(loc='upper right')
ax.grid(True, alpha=0.3)

# Add legend
legend_elements = [Patch(facecolor=event_colors[i], alpha=0.5, 
                         label=f'{event_names[i]} (β_true={true_betas[i]:.1f}, β_fit={betas_fit[i]:.2f})') 
                   for i in range(n_events)]
fig.legend(handles=legend_elements, loc='upper center', ncol=4, fontsize=10, 
           bbox_to_anchor=(0.5, 0.98))

plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.show()

## Summary: R² Across Scenarios

In [ ]:
# Run all scenarios and collect R²
scenarios = [
    ('Baseline\n(all present)', [], []),
    ('Train Missing\nEvent 2', [2], []),
    ('Test Missing\nEvent 2', [], [2]),
    ('Both Missing\nDifferent', [2], [1]),
]

r2_values = []

for name, train_missing, test_missing in scenarios:
    # Create designs
    design_train, _ = create_full_design(n_timepoints, n_events, events_per_run, train_missing, seed=42)
    design_test, _ = create_full_design(n_timepoints, n_events, events_per_run, test_missing, seed=43)
    
    # Generate data
    data_train = generate_fmri_data(design_train, true_betas, noise_std, seed=100)
    data_test = generate_fmri_data(design_test, true_betas, noise_std, seed=101)
    
    # Fit and predict
    betas_fit = fit_ols(data_train, design_train)
    prediction = predict(betas_fit, design_test)
    
    # R²
    r2 = compute_r2(data_test, prediction)
    r2_values.append(r2)

# Plot
fig, ax = plt.subplots(figsize=(10, 6))
x = np.arange(len(scenarios))
bars = ax.bar(x, r2_values, color=['green', 'orange', 'orange', 'red'], alpha=0.7, edgecolor='black')

# Add value labels
for i, (bar, r2) in enumerate(zip(bars, r2_values)):
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height + 0.02,
            f'{r2:.3f}', ha='center', va='bottom', fontweight='bold', fontsize=12)

ax.set_xlabel('Scenario', fontsize=14, fontweight='bold')
ax.set_ylabel('R² (out-of-sample)', fontsize=14, fontweight='bold')
ax.set_title('Cross-Validation R² Across Missing Event Scenarios', fontsize=16, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels([s[0] for s in scenarios], fontsize=11)
ax.set_ylim([0, max(r2_values) * 1.2])
ax.axhline(0, color='black', linewidth=0.8, linestyle='--', alpha=0.5)
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

# Print summary
print("\nSummary:")
print("="*60)
for (name, _, _), r2 in zip(scenarios, r2_values):
    print(f"{name.replace(chr(10), ' '):30s} R² = {r2:.4f}")

## Interpretation

**Expected patterns:**

1. **Baseline**: Highest R² - all events present, can learn and predict all betas

2. **Train Missing Event**: Lower R² - cannot learn beta for missing event, so prediction lacks that component

3. **Test Missing Event**: Similar R² to train missing - fitted beta exists but test data lacks signal, so prediction has extra unexplained variance

4. **Both Missing Different**: Lowest R² - combination of both problems

**Key insight**: The "zero" strategy handles these cases by:
- Using zero beta for events missing from train (can't learn)
- Using fitted beta for events missing from test (still applies, causes mismatch)

The "nuisance" strategy would improve scenario 3 and 4 by projecting out the unpredictable events from test data.